### **WP3 Rapid Trial - Dataset Check and Feature-Selection Screen**

This notebook checks the current WP3.1 matrix, runs compact targeted boosting screens, and saves feature-selection evidence used to rebuild the primary reduced matrix.


##### **Notebook Outline**

- **rapid trial setup:** load the current WP3.1 matrix and define compact screening options.
- **trial model training:** fit quick XGBoost and LightGBM candidates to check current feature-set performance.
- **feature-selection evidence:** calculate validation-screen SHAP evidence and identify features for the reduced matrix.
- **output save:** save only the ranking, audit, and support files needed by WP3.1.

**How to use this notebook**

This notebook is an internal screening step for the current WP3.1 matrix. It uses only the training matrix and a training-only validation split to screen compact targeted boosting models and derive SHAP-guided feature evidence. The held-out test set is reported for context after refitting, but it should not be used to decide which features or configurations to keep.

Run order:
- run WP3.1 until the full and clean matrices exist
- run this notebook to create `rapid_trial_rebuilt_reduced_feature_names`
- rerun the WP3.1 reduced-matrix rebuild cell so later notebooks can use the refreshed `shap_guided_reduced_feature_set`

In [ ]:
#import packages once at the top so the notebook stays easy to rerun
import sys
from pathlib import Path
import time
import warnings
from collections import Counter
import re
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    xgboost_available = True
except Exception as error:
    XGBClassifier = None
    xgboost_available = False
    xgboost_import_error = error

try:
    from lightgbm import LGBMClassifier
    lightgbm_available = True
except Exception as error:
    LGBMClassifier = None
    lightgbm_available = False
    lightgbm_import_error = error

try:
    import shap
    shap_available = True
except Exception as error:
    shap = None
    shap_available = False
    shap_import_error = error

try:
    import matplotlib.pyplot as plt
    matplotlib_available = True
except Exception as error:
    plt = None
    matplotlib_available = False
    matplotlib_import_error = error

print("xgboost available:", xgboost_available)
print("lightgbm available:", lightgbm_available)
print("shap available:", shap_available)

xgboost available: True
lightgbm available: True
shap available: True


**Set Paths And Trial Options**

The path logic works on the Mac project folder and the remote lab copy. The rapid trial uses the cleaned WP3.1 feature-audit matrix by default because this is the best source for rebuilding the SHAP-guided reduced feature set.




In [5]:
#define project paths and compact run options
current_path = Path.cwd().resolve()
project_path = None
for candidate_path in [current_path] + list(current_path.parents):
    if (candidate_path / "code").exists():
        project_path = candidate_path
        break
if project_path is None:
    project_path = Path("/Users/ahthini/Desktop/DissProject")

output_path = project_path / "outputs"
wp3_1_output_path = output_path / "WP3_1"
rapid_trial_output_path = output_path / "WP3_rapid_trial"
trial_table_output_path = rapid_trial_output_path / "tables"
trial_image_output_path = rapid_trial_output_path / "images"
for folder_path in [rapid_trial_output_path, trial_table_output_path, trial_image_output_path]:
    folder_path.mkdir(parents=True, exist_ok=True)

#the source matrix should be the latest cleaned matrix from wp3.1; fall back to full if needed.
source_matrix_variant = "clean_feature_audit"
train_fraction_for_screening = 0.75
random_state = 42
run_xgboost_trials = True
run_lightgbm_trials = True
save_excel_outputs = True
save_csv_outputs = True

#these control the compact shap-guided feature list that wp3.1 will use during its rebuild cell.
max_shap_rows = 2500
target_core_feature_count = 550
family_top_n = 8
always_keep_regex = (
    "previous_|recent_|prior_|psych|self_harm|suicid|depression|substance|aftercare|followup|"
    "discharge|late_|last_order|orders_last|care_team|charlson|elixhauser|hourly|vital|spo2|oxygen|rass|gcs|delirium")

print("project path:", project_path)
print("WP3.1 input folder:", wp3_1_output_path)
print("rapid trial output folder:", rapid_trial_output_path)
print("source matrix variant:", source_matrix_variant)

project path: /scratch/DissProject
WP3.1 input folder: /scratch/DissProject/outputs/WP3_1
rapid trial output folder: /scratch/DissProject/outputs/WP3_rapid_trial
source matrix variant: clean_feature_audit


**Load The Current WP3.1 Matrix**

This cell loads the latest processed feature matrix and prints enough diagnostics to confirm whether a dataset rebuild changed the shape, feature count, or outcome prevalence.




In [6]:
#load matrices, labels, ids, and processed feature names from wp3.1
if source_matrix_variant in ["full", "original", "complete"]:
    matrix_suffix = ""
    X_train_file = wp3_1_output_path / "t3_1_X_train_processed.npz"
    X_test_file = wp3_1_output_path / "t3_1_X_test_processed.npz"
    feature_names_file = wp3_1_output_path / "t3_1_processed_feature_names.csv"
else:
    matrix_suffix = f"_{source_matrix_variant}"
    X_train_file = wp3_1_output_path / f"t3_1_X_train_processed{matrix_suffix}.npz"
    X_test_file = wp3_1_output_path / f"t3_1_X_test_processed{matrix_suffix}.npz"
    feature_names_file = wp3_1_output_path / f"t3_1_processed_feature_names{matrix_suffix}.csv"

if not X_train_file.exists() or not X_test_file.exists() or not feature_names_file.exists():
    print("requested cleaned matrix was not found, falling back to the full processed matrix")
    source_matrix_variant = "full"
    matrix_suffix = ""
    X_train_file = wp3_1_output_path / "t3_1_X_train_processed.npz"
    X_test_file = wp3_1_output_path / "t3_1_X_test_processed.npz"
    feature_names_file = wp3_1_output_path / "t3_1_processed_feature_names.csv"

missing_input_files = [path for path in [X_train_file, X_test_file, feature_names_file,
    wp3_1_output_path / "t3_1_y_train.csv", wp3_1_output_path / "t3_1_y_test.csv"] if not path.exists()]
if missing_input_files:
    raise FileNotFoundError("Missing WP3.1 input files: " + ", ".join(str(path) for path in missing_input_files))

X_train = sparse.load_npz(X_train_file).tocsr()
X_test = sparse.load_npz(X_test_file).tocsr()
y_train = pd.read_csv(wp3_1_output_path / "t3_1_y_train.csv").iloc[:, 0].astype(int).to_numpy()
y_test = pd.read_csv(wp3_1_output_path / "t3_1_y_test.csv").iloc[:, 0].astype(int).to_numpy()
train_ids = pd.read_csv(wp3_1_output_path / "t3_1_train_ids.csv") if (wp3_1_output_path / "t3_1_train_ids.csv").exists() else pd.DataFrame()
test_ids = pd.read_csv(wp3_1_output_path / "t3_1_test_ids.csv") if (wp3_1_output_path / "t3_1_test_ids.csv").exists() else pd.DataFrame()
processed_feature_names = pd.read_csv(feature_names_file).iloc[:, 0].astype(str).tolist()

matrix_diagnostic_df = pd.DataFrame([{
    "source_matrix_variant": source_matrix_variant,
    "X_train_file": str(X_train_file),
    "X_test_file": str(X_test_file),
    "feature_names_file": str(feature_names_file),
    "training_rows": X_train.shape[0],
    "test_rows": X_test.shape[0],
    "processed_feature_count": X_train.shape[1],
    "training_prevalence": float(np.mean(y_train)),
    "test_prevalence": float(np.mean(y_test)),
    "training_nonzero_density": float(X_train.nnz / (X_train.shape[0] * X_train.shape[1])) if X_train.shape[1] else np.nan,
    "test_nonzero_density": float(X_test.nnz / (X_test.shape[0] * X_test.shape[1])) if X_test.shape[1] else np.nan}])

print("WP3.1 matrix diagnostics:")
display(matrix_diagnostic_df.round(4))
print("training outcome counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("test outcome counts:", dict(zip(*np.unique(y_test, return_counts=True))))

WP3.1 matrix diagnostics:


,source_matrix_variant,X_train_file,X_test_file,feature_names_file,training_rows,test_rows,processed_feature_count,training_prevalence,test_prevalence,training_nonzero_density,test_nonzero_density
0,clean_feature_audit,/scratch/DissProject/outputs/WP3_1/t3_1_X_trai...,/scratch/DissProject/outputs/WP3_1/t3_1_X_test...,/scratch/DissProject/outputs/WP3_1/t3_1_proces...,191320,47171,1618,0.1988,0.1976,0.8486,0.8486


training outcome counts: {np.int64(0): np.int64(153281), np.int64(1): np.int64(38039)}
test outcome counts: {np.int64(0): np.int64(37852), np.int64(1): np.int64(9319)}


**Internal Validation Split**

The rapid trial chooses configurations using validation PR-AUC from a training-only split. The held-out test set is reported afterwards as a dataset-level check, not as a tuning target.




In [7]:
#create a training-only validation split for compact hyperparameter screening
train_indices = np.arange(X_train.shape[0])
screen_train_indices, screen_val_indices = train_test_split(train_indices, train_size=train_fraction_for_screening,
    stratify=y_train, random_state=random_state)
X_screen_train = X_train[screen_train_indices]
X_screen_val = X_train[screen_val_indices]
y_screen_train = y_train[screen_train_indices]
y_screen_val = y_train[screen_val_indices]
base_scale_pos_weight = float((y_screen_train == 0).sum() / max((y_screen_train == 1).sum(), 1))

print("screen train shape:", X_screen_train.shape)
print("screen validation shape:", X_screen_val.shape)
print("screen train prevalence:", round(float(y_screen_train.mean()), 4))
print("screen validation prevalence:", round(float(y_screen_val.mean()), 4))
print("base scale_pos_weight:", round(base_scale_pos_weight, 4))

screen train shape: (143490, 1618)
screen validation shape: (47830, 1618)
screen train prevalence: 0.1988
screen validation prevalence: 0.1988
base scale_pos_weight: 4.0296


**Helper Functions**

These helpers keep model screening, threshold reporting, SHAP family grouping, and reduced-feature selection consistent across runs.




In [8]:
#metric, threshold, and shap-family helpers
def find_thresholds(y_true, probability):
    thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
    rows = []
    #loop through each item in this feature block
    for threshold in thresholds:
        pred = (probability >= threshold).astype(int)
        precision = precision_score(y_true, pred, zero_division=0)
        recall = recall_score(y_true, pred, zero_division=0)
        rows.append({"threshold": threshold, "precision": precision, "recall_sensitivity": recall,
            "specificity": recall_score(1 - y_true, 1 - pred, zero_division=0),
            "f1_score": f1_score(y_true, pred, zero_division=0),
            "balanced_precision_recall_gap": abs(precision - recall)})
    threshold_df = pd.DataFrame(rows)
    f1_threshold = float(threshold_df.sort_values(["f1_score", "threshold"], ascending=[False, True]).iloc[0]["threshold"])
    balanced_pr_threshold = float(threshold_df.sort_values(["balanced_precision_recall_gap", "f1_score"], ascending=[True, False]).iloc[0]["threshold"])
    return f1_threshold, balanced_pr_threshold, threshold_df

#evaluate probability model
def evaluate_probability_model(model_name, y_true, probability, threshold):
    pred = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"model": model_name, "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, probability),
        "pr_auc_average_precision": average_precision_score(y_true, probability),
        "brier_score": brier_score_loss(y_true, probability),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, pred, zero_division=0),
        "specificity": tn / max(tn + fp, 1),
        "f1_score": f1_score(y_true, pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "true_positives": int(tp), "false_positives": int(fp),
        "false_negatives": int(fn), "true_negatives": int(tn)}

#define strip processed prefix
def strip_processed_prefix(feature_name):
    for prefix in ["num__", "cat__", "bin__"]:
        if feature_name.startswith(prefix):
            return feature_name[len(prefix):]
    return feature_name

#assign feature family
def assign_feature_family(feature_name):
    name = strip_processed_prefix(str(feature_name)).lower()
    family_rules = [
        ("prior_utilisation", "previous_|recent_|prior_|days_since_previous|admission_gap|utilisation|utilization"),
        ("psychiatric_history", "psych|self_harm|suicid|depression|bipolar|mania|substance|alcohol|opioid|cannabis|stimulant|smi|psychoses"),
        ("discharge_aftercare", "discharge|aftercare|followup|follow_up|home_health|vna|facility|snf|rehab|placement"),
        ("poe_late_order_pathway", "poe|order|late_|last_order|first_order|sitter|observation|elopement|safety|consult|care_team|provider"),
        ("medication_emar_pharmacy", "medication|drug|pharmacy|emar|dose|barcode|infusion|benzodiazepine|antipsychotic|psychotropic|mood_stabil"),
        ("lab_vital_hourly", "lab|labevent|vital|heart|spo2|oxygen|respiratory|systolic|diastolic|temperature|rass|gcs|pain|delirium|urine"),
        ("medical_comorbidity", "charlson|elixhauser|renal|cardio|cancer|diabetes|copd|liver|frailty|weight|bmi|omr|drg|procedure|hcpcs"),
        ("icu_care", "icu|careunit|inputevent|outputevent|procedureevent|datetimeevent|ventilation|pressor"),
        ("demographic_admin", "age|gender|race|language|insurance|marital|admission_type|admission_location|service|transfer|los|hour|weekend|month")]
    for family_name, pattern in family_rules:
        if pd.Series([name]).str.contains(pattern, regex=True).iloc[0]:
            return family_name
    return "other"

#make shap sample indices
def make_shap_sample_indices(y_array, probability, max_rows=2500, seed=42):
    rng = np.random.default_rng(seed)
    row_count = len(y_array)
    if row_count <= max_rows:
        return np.arange(row_count)
    positive_indices = np.where(y_array == 1)[0]
    negative_indices = np.where(y_array == 0)[0]
    high_risk_indices = np.argsort(probability)[-min(500, row_count):]
    selected = set(high_risk_indices.tolist())
    positive_take = min(len(positive_indices), max_rows // 2)
    negative_take = min(len(negative_indices), max_rows - positive_take)
    selected.update(rng.choice(positive_indices, size=positive_take, replace=False).tolist())
    selected.update(rng.choice(negative_indices, size=negative_take, replace=False).tolist())
    selected = np.array(sorted(selected))
    if len(selected) > max_rows:
        selected = rng.choice(selected, size=max_rows, replace=False)
    return np.array(sorted(selected))

**Targeted Trial Configurations**

The list is deliberately short. It is enough to check whether the latest feature rebuild improved the score without turning this notebook into the official full tuning stage.




In [ ]:
#compact targeted configuration sets for rapid dataset testing
targeted_xgboost_trial_configs = [
    {"config_name": "xgb_anchor_depth7_l2_24", "subsample": 0.8, "reg_lambda": 24, "reg_alpha": 1.5,
     "n_estimators": 720, "min_child_weight": 15, "max_depth": 7, "learning_rate": 0.033,
     "gamma": 0.1, "colsample_bytree": 0.65, "scale_pos_weight_multiplier": 0.5},
    {"config_name": "xgb_depth7_cols075_weight035", "subsample": 0.8, "reg_lambda": 24, "reg_alpha": 1.5,
     "n_estimators": 720, "min_child_weight": 15, "max_depth": 7, "learning_rate": 0.033,
     "gamma": 0.1, "colsample_bytree": 0.75, "scale_pos_weight_multiplier": 0.35},
    {"config_name": "xgb_depth6_more_columns", "subsample": 0.85, "reg_lambda": 12, "reg_alpha": 1,
     "n_estimators": 760, "min_child_weight": 10, "max_depth": 6, "learning_rate": 0.035,
     "gamma": 0, "colsample_bytree": 0.75, "scale_pos_weight_multiplier": 0.45}]

targeted_lightgbm_trial_configs = [
    {"config_name": "lgbm_anchor_127_regularised", "subsample": 0.85, "reg_lambda": 35, "reg_alpha": 0.25,
     "num_leaves": 127, "n_estimators": 500, "min_child_samples": 100, "max_depth": -1,
     "learning_rate": 0.025, "colsample_bytree": 0.6, "scale_pos_weight_multiplier": 0.4},
    {"config_name": "lgbm_127_lr0023_500trees", "subsample": 0.85, "reg_lambda": 45, "reg_alpha": 0.5,
     "num_leaves": 127, "n_estimators": 500, "min_child_samples": 100, "max_depth": -1,
     "learning_rate": 0.023, "colsample_bytree": 0.6, "scale_pos_weight_multiplier": 0.35},
    {"config_name": "lgbm_95_l2_60_lr0025", "subsample": 0.85, "reg_lambda": 60, "reg_alpha": 0.5,
     "num_leaves": 95, "n_estimators": 500, "min_child_samples": 100, "max_depth": -1,
     "learning_rate": 0.025, "colsample_bytree": 0.6, "scale_pos_weight_multiplier": 0.4},
    {"config_name": "lgbm_127_weight025_specificity", "subsample": 0.85, "reg_lambda": 45, "reg_alpha": 0.5,
     "num_leaves": 127, "n_estimators": 500, "min_child_samples": 100, "max_depth": -1,
     "learning_rate": 0.025, "colsample_bytree": 0.6, "scale_pos_weight_multiplier": 0.25}]

print("XGBoost trial configs:", len(targeted_xgboost_trial_configs))
print("LightGBM trial configs:", len(targeted_lightgbm_trial_configs))

XGBoost trial configs: 3
LightGBM trial configs: 4


**Train Targeted Trial Models**

Each candidate is screened on the internal validation split. The best configuration from each model family is then refit on the full WP3.1 training matrix and evaluated on the held-out test set.




In [10]:
#train compact targeted xgboost and lightgbm candidates, then refit the best of each family
trial_rows = []
trial_threshold_frames = []
refit_model_registry = []
screen_model_registry = []

#train xgboost config
def train_xgboost_config(config, X_fit, y_fit):
    adjusted_scale_pos_weight = base_scale_pos_weight * config.get("scale_pos_weight_multiplier", 1.0)
    return XGBClassifier(n_estimators=config["n_estimators"], max_depth=config["max_depth"],
        learning_rate=config["learning_rate"], subsample=config["subsample"], colsample_bytree=config["colsample_bytree"],
        min_child_weight=config["min_child_weight"], reg_lambda=config["reg_lambda"], reg_alpha=config["reg_alpha"],
        gamma=config.get("gamma", 0), scale_pos_weight=adjusted_scale_pos_weight, objective="binary:logistic",
        eval_metric="aucpr", tree_method="hist", random_state=random_state, n_jobs=-1)

#train lightgbm config
def train_lightgbm_config(config, X_fit, y_fit):
    adjusted_scale_pos_weight = base_scale_pos_weight * config.get("scale_pos_weight_multiplier", 1.0)
    return LGBMClassifier(n_estimators=config["n_estimators"], num_leaves=config["num_leaves"],
        max_depth=config["max_depth"], learning_rate=config["learning_rate"], subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"], min_child_samples=config["min_child_samples"],
        reg_lambda=config["reg_lambda"], reg_alpha=config["reg_alpha"], scale_pos_weight=adjusted_scale_pos_weight,
        objective="binary", random_state=random_state, n_jobs=-1, verbose=-1)

#define screen and refit candidate
def screen_and_refit_candidate(model_family, config, model_builder):
    model_name = f"Targeted {model_family} - {config['config_name']}"
    print("screening", model_name)
    start_time = time.perf_counter()
    screen_model = model_builder(config, X_screen_train, y_screen_train)
    #fit the model on the current training data
    screen_model.fit(X_screen_train, y_screen_train)
    elapsed_screen_seconds = time.perf_counter() - start_time
    #score admissions as predicted readmission probabilities
    val_probability = screen_model.predict_proba(X_screen_val)[:, 1]
    f1_threshold, balanced_pr_threshold, threshold_df = find_thresholds(y_screen_val, val_probability)
    threshold_df["model"] = model_name
    threshold_df["config_name"] = config["config_name"]
    trial_threshold_frames.append(threshold_df)
    val_metrics = evaluate_probability_model(model_name, y_screen_val, val_probability, f1_threshold)
    val_metrics.update({"model_family": model_family, "config_name": config["config_name"],
        "evaluation_split": "validation_screen", "threshold_role": "validation_f1",
        "balanced_precision_recall_threshold": balanced_pr_threshold,
        "elapsed_training_seconds": elapsed_screen_seconds, "processed_features": X_train.shape[1], **config})
    trial_rows.append(val_metrics)
    screen_model_registry.append({"model_family": model_family, "model_name": model_name,
        "config_name": config["config_name"], "config": config.copy(), "model": screen_model,
        "validation_probability": val_probability, "validation_metrics": val_metrics})
    print("validation PR-AUC:", round(val_metrics["pr_auc_average_precision"], 4),
        "validation ROC-AUC:", round(val_metrics["roc_auc"], 4), "elapsed seconds:", round(elapsed_screen_seconds, 2))
    return screen_model

if run_xgboost_trials and xgboost_available:
    #loop through each item in this feature block
    for config in targeted_xgboost_trial_configs:
        screen_and_refit_candidate("XGBoost", config, train_xgboost_config)
elif run_xgboost_trials:
    print("XGBoost skipped:", xgboost_import_error)

if run_lightgbm_trials and lightgbm_available:
    for config in targeted_lightgbm_trial_configs:
        screen_and_refit_candidate("LightGBM", config, train_lightgbm_config)
elif run_lightgbm_trials:
    print("LightGBM skipped:", lightgbm_import_error)

trial_validation_df = pd.DataFrame(trial_rows).sort_values(
    ["pr_auc_average_precision", "roc_auc"], ascending=False).reset_index(drop=True)
trial_threshold_df = pd.concat(trial_threshold_frames, ignore_index=True) if trial_threshold_frames else pd.DataFrame()
print("validation-screen ranking:")
display(trial_validation_df[["model", "model_family", "config_name", "roc_auc", "pr_auc_average_precision",
    "brier_score", "precision", "recall_sensitivity", "f1_score", "threshold", "balanced_precision_recall_threshold",
    "elapsed_training_seconds"]].round(4))

#refit the best validation configuration from each model family on the full training matrix.
refit_rows = []
#loop through each item in this feature block
for model_family, family_df in trial_validation_df.groupby("model_family", sort=False):
    best_row = family_df.sort_values(["pr_auc_average_precision", "roc_auc"], ascending=False).iloc[0]
    if model_family == "XGBoost":
        config = next(cfg for cfg in targeted_xgboost_trial_configs if cfg["config_name"] == best_row["config_name"])
        model_builder = train_xgboost_config
    elif model_family == "LightGBM":
        config = next(cfg for cfg in targeted_lightgbm_trial_configs if cfg["config_name"] == best_row["config_name"])
        model_builder = train_lightgbm_config
    else:
        continue
    model_name = f"Targeted {model_family} refit - {config['config_name']}"
    print("refitting", model_name)
    start_time = time.perf_counter()
    refit_model = model_builder(config, X_train, y_train)
    refit_model.fit(X_train, y_train)
    elapsed_refit_seconds = time.perf_counter() - start_time
    test_probability = refit_model.predict_proba(X_test)[:, 1]
    test_metrics = evaluate_probability_model(model_name, y_test, test_probability, float(best_row["threshold"]))
    test_metrics.update({"model_family": model_family, "config_name": config["config_name"],
        "evaluation_split": "held_out_test_after_full_refit", "threshold_role": "validation_f1_from_screen",
        "balanced_precision_recall_threshold": float(best_row["balanced_precision_recall_threshold"]),
        "elapsed_training_seconds": elapsed_refit_seconds, "processed_features": X_train.shape[1], **config})
    refit_rows.append(test_metrics)
    refit_model_registry.append({"model_family": model_family, "model_name": model_name, "config_name": config["config_name"],
        "model": refit_model, "test_probability": test_probability, "test_metrics": test_metrics})
    print("test PR-AUC:", round(test_metrics["pr_auc_average_precision"], 4),
        "test ROC-AUC:", round(test_metrics["roc_auc"], 4), "elapsed seconds:", round(elapsed_refit_seconds, 2))

trial_test_df = pd.DataFrame(refit_rows).sort_values(["pr_auc_average_precision", "roc_auc"], ascending=False).reset_index(drop=True)

#keep the strongest full-refit result available for downstream summaries
if not trial_test_df.empty:
    best_refit_row = trial_test_df.iloc[0].copy()
    best_refit_test_pr_auc = float(best_refit_row["pr_auc_average_precision"])
    best_refit_test_roc_auc = float(best_refit_row["roc_auc"])
    best_refit_info = next((item for item in refit_model_registry
        if item["model_family"] == best_refit_row["model_family"]
        and item["config_name"] == best_refit_row["config_name"]), None)
else:
    best_refit_row = pd.Series(dtype="object")
    best_refit_test_pr_auc = np.nan
    best_refit_test_roc_auc = np.nan
    best_refit_info = None

rapid_trial_config_comparison_df = pd.concat([trial_validation_df, trial_test_df], ignore_index=True)
print("held-out test refit ranking:")
display(trial_test_df[["model", "model_family", "config_name", "roc_auc", "pr_auc_average_precision",
    "brier_score", "precision", "recall_sensitivity", "f1_score", "threshold", "processed_features",
    "elapsed_training_seconds"]].round(4))

screening Targeted XGBoost - xgb_anchor_depth7_l2_24
validation PR-AUC: 0.5989 validation ROC-AUC: 0.8039 elapsed seconds: 63.55
screening Targeted XGBoost - xgb_depth7_cols075_weight035
validation PR-AUC: 0.5992 validation ROC-AUC: 0.8034 elapsed seconds: 64.51
screening Targeted XGBoost - xgb_depth6_more_columns
validation PR-AUC: 0.5985 validation ROC-AUC: 0.8033 elapsed seconds: 58.63
screening Targeted LightGBM - lgbm_anchor_127_regularised
validation PR-AUC: 0.6008 validation ROC-AUC: 0.8039 elapsed seconds: 40.29
screening Targeted LightGBM - lgbm_127_lr0023_500trees
validation PR-AUC: 0.6015 validation ROC-AUC: 0.8046 elapsed seconds: 40.69
screening Targeted LightGBM - lgbm_95_l2_60_lr0025
validation PR-AUC: 0.6007 validation ROC-AUC: 0.8049 elapsed seconds: 34.86
screening Targeted LightGBM - lgbm_127_weight025_specificity
validation PR-AUC: 0.6015 validation ROC-AUC: 0.8045 elapsed seconds: 39.52
validation-screen ranking:


,model,model_family,config_name,roc_auc,pr_auc_average_precision,brier_score,precision,recall_sensitivity,f1_score,threshold,balanced_precision_recall_threshold,elapsed_training_seconds
0,Targeted LightGBM - lgbm_127_lr0023_500trees,LightGBM,lgbm_127_lr0023_500trees,0.8046,0.6015,0.1193,0.5199,0.5557,0.5372,0.33,0.99,40.6864
1,Targeted LightGBM - lgbm_127_weight025_specifi...,LightGBM,lgbm_127_weight025_specificity,0.8045,0.6015,0.1175,0.4962,0.5816,0.5355,0.25,0.99,39.5195
2,Targeted LightGBM - lgbm_anchor_127_regularised,LightGBM,lgbm_anchor_127_regularised,0.8039,0.6008,0.1211,0.5014,0.5750,0.5357,0.34,0.99,40.2936
3,Targeted LightGBM - lgbm_95_l2_60_lr0025,LightGBM,lgbm_95_l2_60_lr0025,0.8049,0.6007,0.1215,0.4964,0.5826,0.5361,0.34,0.99,34.8642
4,Targeted XGBoost - xgb_depth7_cols075_weight035,XGBoost,xgb_depth7_cols075_weight035,0.8034,0.5992,0.1197,0.5156,0.5512,0.5328,0.33,0.35,64.5136
5,Targeted XGBoost - xgb_anchor_depth7_l2_24,XGBoost,xgb_anchor_depth7_l2_24,0.8039,0.5989,0.1268,0.5017,0.5701,0.5337,0.39,0.42,63.5515
6,Targeted XGBoost - xgb_depth6_more_columns,XGBoost,xgb_depth6_more_columns,0.8033,0.5985,0.1243,0.5369,0.5334,0.5352,0.40,0.40,58.6268


refitting Targeted LightGBM refit - lgbm_127_lr0023_500trees
test PR-AUC: 0.5958 test ROC-AUC: 0.8006 elapsed seconds: 48.97
refitting Targeted XGBoost refit - xgb_depth7_cols075_weight035
test PR-AUC: 0.5931 test ROC-AUC: 0.7991 elapsed seconds: 83.86
held-out test refit ranking:


,model,model_family,config_name,roc_auc,pr_auc_average_precision,brier_score,precision,recall_sensitivity,f1_score,threshold,processed_features,elapsed_training_seconds
0,Targeted LightGBM refit - lgbm_127_lr0023_500t...,LightGBM,lgbm_127_lr0023_500trees,0.8006,0.5958,0.1195,0.5106,0.5491,0.5291,0.33,1618,48.9671
1,Targeted XGBoost refit - xgb_depth7_cols075_we...,XGBoost,xgb_depth7_cols075_weight035,0.7991,0.5931,0.1199,0.5125,0.5453,0.5284,0.33,1618,83.8561


**Save Compact Trial Score Tables**

Only the dataset diagnostics, validation ranking, test refit ranking, and threshold search are saved. These are enough to check whether a new WP2/WP3.1 dataset changed performance.





In [11]:
#save compact score outputs for dataset comparison
if save_csv_outputs:
    matrix_diagnostic_df.to_csv(trial_table_output_path / "rapid_trial_matrix_diagnostics.csv", index=False)
    trial_validation_df.to_csv(trial_table_output_path / "rapid_trial_validation_config_ranking.csv", index=False)
    trial_test_df.to_csv(trial_table_output_path / "rapid_trial_test_refit_ranking.csv", index=False)
    rapid_trial_config_comparison_df.to_csv(trial_table_output_path / "rapid_trial_config_comparison.csv", index=False)
    if not trial_threshold_df.empty:
        trial_threshold_df.to_csv(trial_table_output_path / "rapid_trial_validation_threshold_search.csv", index=False)

if save_excel_outputs:
    with pd.ExcelWriter(trial_table_output_path / "rapid_trial_model_screening_tables.xlsx") as writer:
        matrix_diagnostic_df.to_excel(writer, sheet_name="matrix_diagnostics", index=False)
        trial_validation_df.to_excel(writer, sheet_name="validation_ranking", index=False)
        trial_test_df.to_excel(writer, sheet_name="test_refit_ranking", index=False)
        if not trial_threshold_df.empty:
            trial_threshold_df.to_excel(writer, sheet_name="threshold_search", index=False)

print("compact rapid-trial score outputs saved to:")
print(trial_table_output_path)

compact rapid-trial score outputs saved to:
/scratch/DissProject/outputs/WP3_rapid_trial/tables


**SHAP review from validation-screen models**

The strongest validation-screen model from each available trial family is explained with SHAP. The global table averages feature influence across these model-family explanations, so the reduced feature list is not driven by held-out test selection or by only one model family.


In [ ]:
#calculate validation-screen shap values for the strongest trial models
if not shap_available:
    raise ImportError("SHAP is not available in this environment.")
if not screen_model_registry:
    raise ValueError("No screened trial models are available for SHAP.")

#pick the best validation-screen model from each family so the feature list is not driven by one model only
best_screen_models_for_shap = []
#loop through each item in this feature block
for model_family, family_df in trial_validation_df.groupby("model_family", sort=False):
    best_row = family_df.sort_values(["pr_auc_average_precision", "roc_auc"], ascending=False).iloc[0]
    matching_models = [item for item in screen_model_registry
        if item["model_family"] == model_family and item["config_name"] == best_row["config_name"]]
    if matching_models:
        best_screen_models_for_shap.append(matching_models[0])

if not best_screen_models_for_shap:
    raise ValueError("No best validation-screen models could be matched for SHAP.")

#keep a named best model for summaries, but build feature evidence from all selected families
best_screen_info = sorted(best_screen_models_for_shap,
    key=lambda item: (item["validation_metrics"]["pr_auc_average_precision"], item["validation_metrics"]["roc_auc"]), reverse=True)[0]
best_trial_model_name = best_screen_info["model_name"]
best_trial_validation_pr_auc = best_screen_info["validation_metrics"]["pr_auc_average_precision"]
best_trial_validation_roc_auc = best_screen_info["validation_metrics"]["roc_auc"]

shap_sample_indices = make_shap_sample_indices(y_screen_val, best_screen_info["validation_probability"], max_rows=max_shap_rows, seed=random_state)
X_shap_sample = X_screen_val[shap_sample_indices]
y_shap_sample = y_screen_val[shap_sample_indices]

print("explained validation-screen model families:", [item["model_family"] for item in best_screen_models_for_shap])
print("top validation-screen model:", best_trial_model_name)
print("SHAP sample rows:", X_shap_sample.shape[0])
print("SHAP sample positives:", int(y_shap_sample.sum()))

#normalise binary shap values
def normalise_binary_shap_values(shap_values_raw, feature_names):
    #shap can return a list, an explanation object, a sparse matrix, or a 3d binary-class array depending on model/library versions.
    expected_feature_count = len(feature_names)

    #define unwrap shap object
    def unwrap_shap_object(values):
        if hasattr(values, "values"):
            values = values.values
        if sparse.issparse(values):
            values = values.toarray()
        values_array = np.asarray(values)
        if values_array.ndim == 0 and values_array.dtype == object:
            return unwrap_shap_object(values_array.item())
        return values_array

    if isinstance(shap_values_raw, (list, tuple)):
        shap_candidates = list(shap_values_raw)
        candidate_order = [1, 0] + list(range(2, len(shap_candidates))) if len(shap_candidates) > 1 else [0]
        last_error = None
        #loop through each item in this feature block
        for candidate_index in candidate_order:
            if candidate_index >= len(shap_candidates):
                continue
            try:
                return normalise_binary_shap_values(shap_candidates[candidate_index], feature_names)
            except ValueError as error:
                last_error = error
        raise last_error if last_error is not None else ValueError("No usable SHAP array was returned.")

    shap_values_processed = unwrap_shap_object(shap_values_raw)

    if shap_values_processed.ndim == 3:
        if shap_values_processed.shape[-1] == 2 and shap_values_processed.shape[1] == expected_feature_count:
            shap_values_processed = shap_values_processed[:, :, 1]
        elif shap_values_processed.shape[0] == 2 and shap_values_processed.shape[2] == expected_feature_count:
            shap_values_processed = shap_values_processed[1, :, :]
        elif shap_values_processed.shape[1] == 2 and shap_values_processed.shape[2] == expected_feature_count:
            shap_values_processed = shap_values_processed[:, 1, :]
        else:
            raise ValueError(f"Unexpected 3D SHAP shape: {shap_values_processed.shape}")

    if shap_values_processed.ndim == 1 and shap_values_processed.shape[0] == expected_feature_count:
        shap_values_processed = shap_values_processed.reshape(1, -1)
    if shap_values_processed.ndim == 2 and shap_values_processed.shape[0] == expected_feature_count and shap_values_processed.shape[1] != expected_feature_count:
        shap_values_processed = shap_values_processed.T
    if shap_values_processed.ndim != 2:
        raise ValueError(f"Expected 2D SHAP values, got shape {shap_values_processed.shape}")
    if shap_values_processed.shape[1] != expected_feature_count:
        raise ValueError(f"SHAP feature count does not match processed feature names: {shap_values_processed.shape[1]} vs {expected_feature_count}")
    return shap_values_processed

shap_frames = []
for model_info in best_screen_models_for_shap:
    model = model_info["model"]
    model_label = model_info["model_name"]
    print("calculating SHAP for:", model_label)
    explainer = shap.TreeExplainer(model)
    try:
        raw_shap_values = explainer.shap_values(X_shap_sample)
    except Exception as sparse_shap_error:
        print("Sparse SHAP calculation failed, retrying with dense matrix:", sparse_shap_error)
        raw_shap_values = explainer.shap_values(X_shap_sample.toarray() if sparse.issparse(X_shap_sample) else X_shap_sample)
    model_shap_values = normalise_binary_shap_values(raw_shap_values, processed_feature_names)
    model_mean_abs_shap = np.abs(model_shap_values).mean(axis=0)
    model_mean_signed_shap = model_shap_values.mean(axis=0)
    shap_frames.append(pd.DataFrame({"processed_feature_name": processed_feature_names,
        "model_family": model_info["model_family"], "model_name": model_label,
        "model_mean_abs_shap": model_mean_abs_shap, "model_mean_signed_shap": model_mean_signed_shap}))

#summarise row-level records to admission-level features
rapid_trial_model_shap_importance_df = pd.concat(shap_frames, ignore_index=True)
shap_global_importance_df = (rapid_trial_model_shap_importance_df.groupby("processed_feature_name", as_index=False)
    .agg(mean_abs_shap=("model_mean_abs_shap", "mean"), max_abs_shap=("model_mean_abs_shap", "max"),
        mean_signed_shap=("model_mean_signed_shap", "mean"), shap_model_count=("model_name", "nunique")))
shap_global_importance_df["feature_name_clean"] = shap_global_importance_df["processed_feature_name"].apply(strip_processed_prefix)
shap_global_importance_df["feature_family"] = shap_global_importance_df["processed_feature_name"].apply(assign_feature_family)
shap_global_importance_df["nonzero_train_percent"] = np.asarray((X_train != 0).sum(axis=0)).ravel() / X_train.shape[0] * 100
shap_global_importance_df = shap_global_importance_df.sort_values(
    ["mean_abs_shap", "max_abs_shap"], ascending=False).reset_index(drop=True)
shap_global_importance_df["importance_rank"] = np.arange(1, len(shap_global_importance_df) + 1)

shap_feature_family_importance_df = (shap_global_importance_df.groupby("feature_family", as_index=False)
    .agg(total_mean_abs_shap=("mean_abs_shap", "sum"), mean_abs_shap=("mean_abs_shap", "mean"),
        top_feature_mean_abs_shap=("mean_abs_shap", "max"), feature_count=("processed_feature_name", "count"))
    .sort_values("total_mean_abs_shap", ascending=False).reset_index(drop=True))
shap_feature_family_importance_df["family_importance_rank"] = np.arange(1, len(shap_feature_family_importance_df) + 1)

rapid_trial_top5_features_by_family_df = (shap_global_importance_df.sort_values(["feature_family", "mean_abs_shap"], ascending=[True, False])
    .groupby("feature_family", as_index=False).head(5).reset_index(drop=True))

if save_csv_outputs:
    #save this table or artefact for later review
    shap_global_importance_df.to_csv(trial_table_output_path / "rapid_trial_shap_global_importance.csv", index=False)
    rapid_trial_model_shap_importance_df.to_csv(trial_table_output_path / "rapid_trial_model_family_shap_importance.csv", index=False)
    shap_feature_family_importance_df.to_csv(trial_table_output_path / "rapid_trial_feature_family_importance.csv", index=False)
    rapid_trial_top5_features_by_family_df.to_csv(trial_table_output_path / "rapid_trial_top5_features_by_family.csv", index=False)

if save_excel_outputs:
    shap_global_importance_df.to_excel(trial_table_output_path / "rapid_trial_shap_global_importance.xlsx", index=False)
    rapid_trial_model_shap_importance_df.to_excel(trial_table_output_path / "rapid_trial_model_family_shap_importance.xlsx", index=False)
    shap_feature_family_importance_df.to_excel(trial_table_output_path / "rapid_trial_feature_family_importance.xlsx", index=False)
    rapid_trial_top5_features_by_family_df.to_excel(trial_table_output_path / "rapid_trial_top5_features_by_family.xlsx", index=False)

print("top global SHAP features from validation-screen models:")
display(shap_global_importance_df.head(30).round(5))
print("top SHAP feature families:")
display(shap_feature_family_importance_df.round(5))

explained validation-screen model families: ['LightGBM', 'XGBoost']
top validation-screen model: Targeted LightGBM - lgbm_127_lr0023_500trees
SHAP sample rows: 2500
SHAP sample positives: 1427
calculating SHAP for: Targeted LightGBM - lgbm_127_lr0023_500trees
calculating SHAP for: Targeted XGBoost - xgb_depth7_cols075_weight035
top global SHAP features from validation-screen models:


,processed_feature_name,mean_abs_shap,max_abs_shap,mean_signed_shap,shap_model_count,feature_name_clean,feature_family,nonzero_train_percent,importance_rank
0,num__late_orders_without_prior_psych_history_flag,0.34558,0.40782,0.15413,2,late_orders_without_prior_psych_history_flag,prior_utilisation,100.00000,1
1,num__late_order_severity_per_late_order_last_12h,0.13142,0.14821,0.08401,2,late_order_severity_per_late_order_last_12h,poe_late_order_pathway,100.00000,2
2,num__last_order_close_to_discharge_hours,0.13019,0.13509,0.10283,2,last_order_close_to_discharge_hours,discharge_aftercare,100.00000,3
3,num__orders_last_12h_before_discharge,0.09607,0.10696,0.06339,2,orders_last_12h_before_discharge,discharge_aftercare,100.00000,4
4,num__low_order_activity_with_self_harm_flag,0.09393,0.10510,0.05727,2,low_order_activity_with_self_harm_flag,psychiatric_history,100.00000,5
5,cat__discharge_location_Unknown / not modelled,0.08921,0.10559,0.01035,2,discharge_location_Unknown / not modelled,discharge_aftercare,100.00000,6
6,num__previous_psych_admissions_365d,0.07519,0.07968,-0.00216,2,previous_psych_admissions_365d,prior_utilisation,100.00000,7
7,num__previous_psych_admissions_180d,0.07437,0.07898,-0.00090,2,previous_psych_admissions_180d,prior_utilisation,100.00000,8
8,num__unmanaged_psychiatric_discharge_risk_score,0.05009,0.05492,0.01507,2,unmanaged_psychiatric_discharge_risk_score,psychiatric_history,5.94606,9
9,num__discharge_hour,0.04503,0.04619,0.02308,2,discharge_hour,discharge_aftercare,100.00000,10


top SHAP feature families:


,feature_family,total_mean_abs_shap,mean_abs_shap,top_feature_mean_abs_shap,feature_count,family_importance_rank
0,prior_utilisation,0.90146,0.00567,0.34558,159,1
1,discharge_aftercare,0.71081,0.00508,0.13019,140,2
2,psychiatric_history,0.67890,0.00409,0.09393,166,3
3,lab_vital_hourly,0.62438,0.00127,0.01918,492,4
4,poe_late_order_pathway,0.59023,0.00404,0.13142,146,5
5,demographic_admin,0.32606,0.00162,0.03223,201,6
6,other,0.29570,0.00242,0.03709,122,7
7,medical_comorbidity,0.14855,0.00183,0.02436,81,8
8,medication_emar_pharmacy,0.12402,0.00264,0.02556,47,9
9,icu_care,0.03608,0.00056,0.02063,64,10


**Build the SHAP-guided feature list for WP3.1**

This cell writes the feature-name list that WP3.1 reads when rebuilding `shap_guided_reduced_feature_set`. The list is based on validation-screen SHAP ranking plus a small family-level safety net so important clinical families are not lost just because their signal is distributed across several variables.


In [13]:
#create the compact shap-guided feature list consumed by wp3.1
core_top_features = shap_global_importance_df.head(target_core_feature_count)["processed_feature_name"].tolist()
family_top_features = (shap_global_importance_df.sort_values(["feature_family", "mean_abs_shap"], ascending=[True, False])
    .groupby("feature_family", as_index=False).head(family_top_n)["processed_feature_name"].tolist())

always_keep_candidates = shap_global_importance_df.loc[
    shap_global_importance_df["processed_feature_name"].str.contains(always_keep_regex, case=False, regex=True, na=False)
    & shap_global_importance_df["mean_abs_shap"].gt(0), "processed_feature_name"].tolist()

selected_feature_names = []
#loop through each item in this feature block
for feature_name in core_top_features + family_top_features + always_keep_candidates:
    if feature_name not in selected_feature_names:
        selected_feature_names.append(feature_name)

rebuilt_reduced_feature_names_df = shap_global_importance_df.loc[
    shap_global_importance_df["processed_feature_name"].isin(selected_feature_names),
    ["processed_feature_name", "feature_name_clean", "feature_family", "importance_rank", "mean_abs_shap",
        "mean_signed_shap", "nonzero_train_percent"]].copy()
rebuilt_reduced_feature_names_df["selection_reason"] = np.select([
        rebuilt_reduced_feature_names_df["processed_feature_name"].isin(core_top_features),
        rebuilt_reduced_feature_names_df["processed_feature_name"].isin(family_top_features),
        rebuilt_reduced_feature_names_df["processed_feature_name"].isin(always_keep_candidates)],
    ["top_global_shap", "top_within_family", "clinical_pattern_addback"], default="selected")
rebuilt_reduced_feature_names_df = rebuilt_reduced_feature_names_df.sort_values(
    ["importance_rank", "processed_feature_name"]).reset_index(drop=True)

rapid_trial_rebuild_summary_df = pd.DataFrame([{
    "source_matrix_variant": source_matrix_variant,
    "source_processed_feature_count": len(processed_feature_names),
    "target_core_feature_count": target_core_feature_count,
    "family_top_n": family_top_n,
    "selected_feature_count_for_wp3_1_rebuild": len(rebuilt_reduced_feature_names_df),
    "explained_model": best_trial_model_name,
    "explained_model_selection_basis": "best_validation_screen_pr_auc",
    "explained_model_validation_pr_auc_average_precision": best_trial_validation_pr_auc,
    "explained_model_validation_roc_auc": best_trial_validation_roc_auc,
    "best_refit_test_pr_auc_average_precision": best_refit_test_pr_auc,
    "best_refit_test_roc_auc": best_refit_test_roc_auc}])

rebuilt_feature_list_path = trial_table_output_path / "rapid_trial_rebuilt_reduced_feature_names.xlsx"
rebuilt_feature_list_csv_path = trial_table_output_path / "rapid_trial_rebuilt_reduced_feature_names.csv"
#save this table or artefact for later review
rebuilt_reduced_feature_names_df.to_excel(rebuilt_feature_list_path, index=False)
if save_csv_outputs:
    #save this table or artefact for later review
    rebuilt_reduced_feature_names_df.to_csv(rebuilt_feature_list_csv_path, index=False)
    rapid_trial_rebuild_summary_df.to_csv(trial_table_output_path / "rapid_trial_rebuilt_reduced_feature_summary.csv", index=False)

print("SHAP-guided feature list saved for WP3.1 rebuild:")
print(rebuilt_feature_list_path)
print("selected feature count:", len(rebuilt_reduced_feature_names_df))
display(rapid_trial_rebuild_summary_df.round(4))
display(rebuilt_reduced_feature_names_df.head(30).round(5))

SHAP-guided feature list saved for WP3.1 rebuild:
/scratch/DissProject/outputs/WP3_rapid_trial/tables/rapid_trial_rebuilt_reduced_feature_names.xlsx
selected feature count: 904


,source_matrix_variant,source_processed_feature_count,target_core_feature_count,family_top_n,selected_feature_count_for_wp3_1_rebuild,explained_model,explained_model_selection_basis,explained_model_validation_pr_auc_average_precision,explained_model_validation_roc_auc,best_refit_test_pr_auc_average_precision,best_refit_test_roc_auc
0,clean_feature_audit,1618,550,8,904,Targeted LightGBM - lgbm_127_lr0023_500trees,best_validation_screen_pr_auc,0.6015,0.8046,0.5958,0.8006


,processed_feature_name,feature_name_clean,feature_family,importance_rank,mean_abs_shap,mean_signed_shap,nonzero_train_percent,selection_reason
0,num__late_orders_without_prior_psych_history_flag,late_orders_without_prior_psych_history_flag,prior_utilisation,1,0.34558,0.15413,100.00000,top_global_shap
1,num__late_order_severity_per_late_order_last_12h,late_order_severity_per_late_order_last_12h,poe_late_order_pathway,2,0.13142,0.08401,100.00000,top_global_shap
2,num__last_order_close_to_discharge_hours,last_order_close_to_discharge_hours,discharge_aftercare,3,0.13019,0.10283,100.00000,top_global_shap
3,num__orders_last_12h_before_discharge,orders_last_12h_before_discharge,discharge_aftercare,4,0.09607,0.06339,100.00000,top_global_shap
4,num__low_order_activity_with_self_harm_flag,low_order_activity_with_self_harm_flag,psychiatric_history,5,0.09393,0.05727,100.00000,top_global_shap
5,cat__discharge_location_Unknown / not modelled,discharge_location_Unknown / not modelled,discharge_aftercare,6,0.08921,0.01035,100.00000,top_global_shap
6,num__previous_psych_admissions_365d,previous_psych_admissions_365d,prior_utilisation,7,0.07519,-0.00216,100.00000,top_global_shap
7,num__previous_psych_admissions_180d,previous_psych_admissions_180d,prior_utilisation,8,0.07437,-0.00090,100.00000,top_global_shap
8,num__unmanaged_psychiatric_discharge_risk_score,unmanaged_psychiatric_discharge_risk_score,psychiatric_history,9,0.05009,0.01507,5.94606,top_global_shap
9,num__discharge_hour,discharge_hour,discharge_aftercare,10,0.04503,0.02308,100.00000,top_global_shap


**Optional SHAP Plots**

These plots are lightweight visual checks. They are not required by WP3.1, but they help confirm that the rapid trial is relying on clinically sensible feature families.




In [14]:
#save compact shap plots for visual review
if matplotlib_available:
    top_plot_df = shap_global_importance_df.head(30).sort_values("mean_abs_shap", ascending=True)
    plt.figure(figsize=(9, 7))
    plt.barh(top_plot_df["feature_name_clean"], top_plot_df["mean_abs_shap"], color="#2f6f9f")
    plt.xlabel("mean absolute SHAP value")
    plt.title("rapid-trial top SHAP features")
    plt.tight_layout()
    plt.savefig(trial_image_output_path / "rapid_trial_top30_shap_features.png", dpi=250)
    plt.close()

    family_plot_df = shap_feature_family_importance_df.sort_values("total_mean_abs_shap", ascending=True)
    plt.figure(figsize=(8, 5))
    plt.barh(family_plot_df["feature_family"], family_plot_df["total_mean_abs_shap"], color="#4c956c")
    plt.xlabel("total mean absolute SHAP value")
    plt.title("rapid-trial SHAP feature-family importance")
    plt.tight_layout()
    plt.savefig(trial_image_output_path / "rapid_trial_feature_family_shap_importance.png", dpi=250)
    plt.close()
    print("SHAP plots saved to:", trial_image_output_path)
else:
    print("plotting skipped because matplotlib is unavailable")

SHAP plots saved to: /scratch/DissProject/outputs/WP3_rapid_trial/images


**End-Of-Run Summary**

This final cell prints the files that matter. After this notebook runs, rerun WP3.1 from the feature-audit/rebuild section so it can consume `rapid_trial_rebuilt_reduced_feature_names.xlsx` and write the refreshed `shap_guided_reduced_feature_set` matrices.




In [15]:
#print the short rerun summary
summary_lines = ["rapid trial complete",
    f"source matrix: {source_matrix_variant}",
    f"processed features tested: {len(processed_feature_names)}",
    f"best explained validation-screen model: {best_trial_model_name}",
    f"best explained validation PR-AUC: {best_trial_validation_pr_auc:.4f}",
    f"best refit test PR-AUC: {best_refit_test_pr_auc:.4f}",
    f"best refit test ROC-AUC: {best_refit_test_roc_auc:.4f}",
    f"feature list for WP3.1: {rebuilt_feature_list_path}",
    "rerun WP3.1 rebuild cells to create the refreshed shap_guided_reduced_feature_set matrices"]
run_summary_path = rapid_trial_output_path / "rapid_trial_run_summary.txt"
run_summary_path.write_text("\n".join(summary_lines))
for line in summary_lines:
    print(line)

output_file_summary_df = pd.DataFrame([{"file_name": path.name, "folder": str(path.parent),
    "file_size_mb": round(path.stat().st_size / (1024 ** 2), 3)} for path in sorted(rapid_trial_output_path.rglob("*")) if path.is_file()])
output_file_summary_df.to_csv(trial_table_output_path / "rapid_trial_output_file_summary.csv", index=False)
display(output_file_summary_df)

rapid trial complete
source matrix: clean_feature_audit
processed features tested: 1618
best explained validation-screen model: Targeted LightGBM - lgbm_127_lr0023_500trees
best explained validation PR-AUC: 0.6015
best refit test PR-AUC: 0.5958
best refit test ROC-AUC: 0.8006
feature list for WP3.1: /scratch/DissProject/outputs/WP3_rapid_trial/tables/rapid_trial_rebuilt_reduced_feature_names.xlsx
rerun WP3.1 rebuild cells to create the refreshed shap_guided_reduced_feature_set matrices


,file_name,folder,file_size_mb
0,rapid_trial_feature_family_shap_importance.png,/scratch/DissProject/outputs/WP3_rapid_trial/i...,0.095
1,rapid_trial_top30_shap_features.png,/scratch/DissProject/outputs/WP3_rapid_trial/i...,0.337
2,rapid_trial_run_summary.txt,/scratch/DissProject/outputs/WP3_rapid_trial,0.000
3,rapid_trial_config_comparison.csv,/scratch/DissProject/outputs/WP3_rapid_trial/t...,0.004
4,rapid_trial_feature_family_importance.csv,/scratch/DissProject/outputs/WP3_rapid_trial/t...,0.001
5,rapid_trial_feature_family_importance.xlsx,/scratch/DissProject/outputs/WP3_rapid_trial/t...,0.006
6,rapid_trial_matrix_diagnostics.csv,/scratch/DissProject/outputs/WP3_rapid_trial/t...,0.001
7,rapid_trial_model_family_shap_importance.csv,/scratch/DissProject/outputs/WP3_rapid_trial/t...,0.388
8,rapid_trial_model_family_shap_importance.xlsx,/scratch/DissProject/outputs/WP3_rapid_trial/t...,0.132
9,rapid_trial_model_screening_tables.xlsx,/scratch/DissProject/outputs/WP3_rapid_trial/t...,0.057


**CSV Table Workbook**

This cell keeps the individual CSV files and also creates one Excel workbook where each CSV table is available as a separate sheet. The workbook is for easier review and sharing; the CSV files remain the primary machine-readable outputs.


In [16]:
#compile csv tables into one workbook while preserving the original csv files
possible_output_roots = []
#loop through each item in this feature block
for output_name in ['rapid_trial_output_path']:
    if output_name in globals():
        output_root = globals()[output_name]
        if isinstance(output_root, Path) and output_root.exists() and output_root not in possible_output_roots:
            possible_output_roots.append(output_root)

if not possible_output_roots:
    print("no wp3 output roots were available for workbook compilation")
else:
    primary_output_root = possible_output_roots[0]
    workbook_path = primary_output_root / f"{primary_output_root.name.lower()}_csv_table_workbook.xlsx"
    csv_paths = []
    #loop through each item in this feature block
    for output_root in possible_output_roots:
        csv_paths.extend(sorted(path for path in output_root.rglob("*.csv") if path.is_file()))
    csv_paths = sorted(dict.fromkeys(csv_paths))

    #make sheet name
    def make_sheet_name(csv_path, used_names):
        relative_name = csv_path.relative_to(primary_output_root.parent).with_suffix("").as_posix()
        relative_name = re.sub(r"[^0-9a-zA-Z]+", "_", relative_name).strip("_").lower()
        parts = [part for part in relative_name.split("_") if part not in ["outputs", primary_output_root.name.lower(), "csv", "tables"]]
        base_name = "_".join(parts[-4:]) if parts else csv_path.stem.lower()
        base_name = base_name[:31] or "table"
        sheet_name = base_name
        counter = 1
        while sheet_name in used_names:
            suffix = f"_{counter}"
            sheet_name = base_name[:31 - len(suffix)] + suffix
            counter += 1
        used_names.add(sheet_name)
        return sheet_name

    workbook_index_rows = []
    used_sheet_names = set()
    max_excel_rows = 200000
    with pd.ExcelWriter(workbook_path) as writer:
        #loop through each item in this feature block
        for csv_path in csv_paths:
            try:
                table_df = pd.read_csv(csv_path)
            except Exception as error:
                workbook_index_rows.append({"csv_path": str(csv_path), "sheet_name": "", "rows": np.nan,
                    "columns": np.nan, "rows_written": 0, "note": f"read_failed: {error}"})
                continue
            sheet_name = make_sheet_name(csv_path, used_sheet_names)
            rows_written = min(len(table_df), max_excel_rows)
            #save this table or artefact for later review
            table_df.head(max_excel_rows).to_excel(writer, sheet_name=sheet_name, index=False)
            workbook_index_rows.append({"csv_path": str(csv_path), "sheet_name": sheet_name, "rows": len(table_df),
                "columns": table_df.shape[1], "rows_written": rows_written,
                "note": "truncated_for_excel" if len(table_df) > max_excel_rows else "complete"})
        workbook_index_df = pd.DataFrame(workbook_index_rows)
        #save this table or artefact for later review
        workbook_index_df.to_excel(writer, sheet_name="workbook_index", index=False)
    #save this table or artefact for later review
    workbook_index_df.to_csv(primary_output_root / f"{primary_output_root.name.lower()}_csv_table_workbook_index.csv", index=False)
    print("csv table workbook saved to:")
    print(workbook_path)
    display(workbook_index_df)

csv table workbook saved to:
/scratch/DissProject/outputs/WP3_rapid_trial/wp3_rapid_trial_csv_table_workbook.xlsx


,csv_path,sheet_name,rows,columns,rows_written,note
0,/scratch/DissProject/outputs/WP3_rapid_trial/t...,rapid_trial_config_comparison,9,33,9,complete
1,/scratch/DissProject/outputs/WP3_rapid_trial/t...,trial_feature_family_importance,10,6,10,complete
2,/scratch/DissProject/outputs/WP3_rapid_trial/t...,rapid_trial_matrix_diagnostics,1,11,1,complete
3,/scratch/DissProject/outputs/WP3_rapid_trial/t...,model_family_shap_importance,3236,5,3236,complete
4,/scratch/DissProject/outputs/WP3_rapid_trial/t...,trial_output_file_summary,23,3,23,complete
5,/scratch/DissProject/outputs/WP3_rapid_trial/t...,rebuilt_reduced_feature_names,904,8,904,complete
6,/scratch/DissProject/outputs/WP3_rapid_trial/t...,rebuilt_reduced_feature_summary,1,11,1,complete
7,/scratch/DissProject/outputs/WP3_rapid_trial/t...,trial_shap_global_importance,1618,9,1618,complete
8,/scratch/DissProject/outputs/WP3_rapid_trial/t...,trial_test_refit_ranking,2,33,2,complete
9,/scratch/DissProject/outputs/WP3_rapid_trial/t...,top5_features_by_family,50,9,50,complete
